In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

Saving training_df.tsv to training_df.tsv


In [ ]:
def benchmark_pos(fasta, metadata):
  meta = pd.read_csv(metadata, sep = "\t", names = ["UniProt_ID", "Species", "Kingdom", "Length", "Cleavage", "Fold"])
  IDs = []
  fragments = []
  sp15 = []
  cl = []
  lab = []
  pred = []
  cleav = []
  length = []
  tm = []
  with open(fasta, "r") as r:
    for i in range(5):
      for line in r:
        if line.startswith(">"):
          id = line[1:-1]
          IDs.append(id)
          cl.append(int(1))
          lab.append(int(0))
          pred.append(int(0))
          length.append(meta[meta["UniProt_ID"] == id]["Length"].iloc[0])
          cleavage = int(meta[meta["UniProt_ID"] == id]["Cleavage"].iloc[0])
          tm.append(False)
        elif len(line)>90:
          fragments.append(line[:90])
          cleav.append(int(cleavage))
          sp15.append(line[cleavage-13:cleavage+2])
        else:
          fragments.append(line)
    IDs = pd.Series(IDs)
    fragments = pd.Series(fragments[1:])
    sp15 = pd.Series(sp15)
    cl = pd.Series(cl)
    lab = pd.Series(lab)
    pred = pd.Series(pred)
    cleav = pd.Series(cleav)
    length = pd.Series(length)
    tm = pd.Series(tm)
    benchmark = pd.concat([IDs, cl, lab, fragments, sp15, pred, cleav, length, tm], axis=1)
    benchmark = benchmark.rename(columns = {0:"UniProt_ID", 1:"Class", 2:"Label", 3:"Frag_90", 4:"SP_15", 5:"Prediction", 6:"Cleavage", 7:"Seq_Length", 8:"TM_Helix" })
  return benchmark

def benchmark_neg(fasta, metadata):
  meta = pd.read_csv(metadata, sep = "\t", names = ["UniProt_ID", "Species", "Kingdom", "Length", "TM_Helix", "Fold"])
  IDs = []
  fragments = []
  sp15 = []
  cl = []
  lab = []
  pred = []
  cleav = []
  length = []
  tm = []
  with open(fasta, "r") as r:
    for i in range(5):
      for line in r:
        if line.startswith(">"):
          id = line[1:-1]
          IDs.append(id)
          cl.append(int(1))
          lab.append(int(0))
          pred.append(int(0))
          length.append(meta[meta["UniProt_ID"] == id]["Length"].iloc[0])
          tm.append(meta[meta["UniProt_ID"] == id]["TM_Helix"].iloc[0])
          cleav.append(int(0))
          sp15.append("NIL")
        elif len(line)>90:
          fragments.append(line[:90])
        else:
          fragments.append(line)
    IDs = pd.Series(IDs)
    fragments = pd.Series(fragments[1:])
    sp15 = pd.Series(sp15)
    cl = pd.Series(cl)
    lab = pd.Series(lab)
    pred = pd.Series(pred)
    cleav = pd.Series(cleav)
    length = pd.Series(length)
    tm = pd.Series(tm)
    benchmark = pd.concat([IDs, cl, lab, fragments, sp15, pred, cleav, length, tm], axis=1)
    benchmark = benchmark.rename(columns = {0:"UniProt_ID", 1:"Class", 2:"Label", 3:"Frag_90", 4:"SP_15", 5:"Prediction", 6:"Cleavage", 7:"Seq_Length", 8: "TM_Helix" })
  return benchmark


In [ ]:
pos = benchmark_pos("pos_bench_clean.fasta","pos_dss.tsv")
neg = benchmark_neg("neg_bench_clean.fasta","neg_dss.tsv")

benchmark_data = pd.concat([pos, neg])

benchmark_data.to_csv("benchmarking.tsv", sep = "\t")



In [ ]:
df = pd.read_csv("training_df.tsv", sep = "\t")

meta_pos = pd.read_csv("pos_dss.tsv", sep = "\t", names = ["UniProt_ID", "Species", "Kingdom", "Length", "Cleavage", "Fold"])
meta_neg = pd.read_csv("neg_dss.tsv", sep = "\t", names = ["UniProt_ID", "Species", "Kingdom", "Length", "TM_Helix", "Fold"])
pred = []
cleav = []
leng = []
tm = []
i = 0
for index,row in df.iterrows():
  pred.append(0)
  if row['Class'] == 1:
    cleav.append(meta_pos["Cleavage"][i])
    leng.append(meta_pos["Length"][i])
    tm.append(False)
  elif row['Class'] == 0:
    cleav.append(0)
    leng.append(meta_neg["Length"][i])
    tm.append(meta_neg["TM_Helix"][i])
  i += 1
pred = pd.Series(pred)
cleav = pd.Series(cleav)
leng = pd.Series(leng)
tm = pd.Series(tm)
final = pd.concat([df,pred,cleav,leng,tm], axis = 1)
final = final.rename(columns = {"UniProt_ID":"UniProt_ID", "Class":"Class", "Label":"Label", "Frag_90":"Frag_90", "SP_15":"SP_15", 0:"Prediction", 1:"Cleavage", 2:"Seq_Length", 3: "TM_Helix" })
final.to_csv("Performance.tsv", sep = "\t", header= True)